# 09 · Hosted agents – when configuration is not enough

A **prompt agent** is configuration. A **hosted agent** is *your code* – any framework, any library, custom logic – packaged and run by Foundry with an endpoint, an Entra identity, scaling, tracing and session storage.

| | 🧾 Prompt agent | 🏗️ Hosted agent |
|---|---|---|
| You write | definition (model, instructions, tools) | code (Agent Framework, LangGraph, your own) |
| Runs | inside Foundry's agent runtime | in a managed container/sandbox |
| Tools | built-in, MCP, A2A, functions you loop yourself | anything your code can call |
| Deploy | `create_version` | `azd deploy` (code ZIP or container) |
| Protocols | Responses | Responses, Invocations, A2A, ... |

⚠️ Deploying needs the **Foundry Project Manager** role on the project (formerly *Azure AI Project Manager*). `python3 scripts/grant_hosted_role.py` assigns it to you. Hosted agents are a newer feature: check the [docs](https://learn.microsoft.com/azure/foundry/agents/quickstarts/quickstart-hosted-agent) if a command differs.

In [ ]:
import sys; sys.path.insert(0, '.')
from stark_helpers import *

print((ROOT / "src" / "jarvis-hosted" / "main.py").read_text())

## What the code does
1. `FoundryChatClient` talks to the model through the project (identity: yours locally, the **agent identity** when hosted).
2. `Agent` adds instructions, the local tool `workshop_status` and the skills from `skills/` (direct injection, chapter 07).
3. `ResponsesHostServer` exposes the agent over the **Responses protocol**: Foundry stores conversations and handles streaming.

## 1 · Install the azd extension and run locally
```bash
python3 scripts/grant_hosted_role.py         # once: Foundry Project Manager role for you
azd extension install microsoft.foundry      # adds: azd ai agent ...
python3 scripts/sync_skills.py               # copy skills/ into src/jarvis-hosted/skills
azd ai agent run                             # starts main.py locally and opens the agent inspector
azd ai agent invoke --local "Status of the arc reactor?"
```

## 2 · Deploy
`azd deploy` zips `src/jarvis-hosted`, Foundry builds it remotely (`codeConfiguration` in [azure.yaml](../azure.yaml)) and creates version 1. No Docker needed in code mode.

In [ ]:
sh("azd deploy jarvis-hosted --no-prompt")

## 3 · Call it like any agent
Hosted agents answer the same Responses API as prompt agents.

In [ ]:
hosted = "jarvis-hosted"
oai = openai_client(agent_name=hosted)
conv = oai.conversations.create()
for q in ["What is the status of the arc reactor?", "A suit just started smoking in the hangar. What do I do?"]:
    r = oai.responses.create(conversation=conv.id, input=q)
    print(f"> {q}\n{r.output_text}\n")

The second question should follow the **stark-escalation** skill ("Severity: critical", evacuate, notify Pepper) – proof the skill reached the container.

## 4 · Observe it
```bash
azd ai agent monitor --follow     # container logs
azd ai agent show                 # versions, endpoint, protocols
```
Hosted agents get `APPLICATIONINSIGHTS_CONNECTION_STRING` from the platform, so their traces land in the same Application Insights as chapter 03 – find the conversation id in **Traces**.

## 5 · Clean up

In [ ]:
oai.conversations.delete(conv.id)
# delete_agent("jarvis-hosted")   # uncomment to remove the hosted agent; `azd down --purge` removes everything

## Recap
Prompt agents for speed and simplicity; hosted agents when you need code. Both are first-class Foundry agents: same conversations, tracing, and evaluations (next act).

> 💡 Next: Act III – **F.R.I.D.A.Y.** proves it works. [10 · Datasets and evaluations](10_datasets_and_evals.ipynb).